# OCR_pdf v8.0 — GPU OCR → Keyword → Excel trên Kaggle

Notebook chạy [VuTuanAn1403/OCR](https://github.com/VuTuanAn1403/OCR)
theo **pipeline v8.0**: OCR mọi trang trên CUDA (`VietOCR vgg_transformer` + `PP-DocLayout_plus-L` ONNX), dựng bảng/công thức bằng Docling,
xuất Excel riêng rồi xác nhận ma trận chung trước khi chuyển sang PDF tiếp theo.

1. Import file `.ipynb` vào Kaggle, bật **Internet**, chọn **GPU** và **Add Input** dataset PDF.
2. Chạy các cell theo thứ tự. Mặc định lấy **tất cả PDF**, kể cả trong thư mục con, không giới hạn 50 file.
3. Lượt đầu cài Python 3.11, hai worker GPU và tải nhiều GB trọng số; kernel Kaggle không cài Torch/Docling của dự án.
4. Kết quả ở `/kaggle/working/ocr_pdf_v80_results`; tải Excel hoặc ZIP ở cell cuối / mục Output.

Một worker dùng GPU 0; notebook không tự chia công việc qua hai GPU. **v8.0 yêu cầu NVIDIA CUDA**;
nếu cần CPU hoặc chọn một vài trang, dùng notebook `benchmark/baselines/ocr_pdf_legacy_kaggle.ipynb` (legacy).
Để thử v8.0 với một tài liệu, đặt `FILE_LIMIT = 1` và dùng thư mục `RESULTS_DIR` riêng.

**Bản sửa lỗi Kaggle 17/10/2026:** repository có sẵn font Noto Sans cho PDF trung gian
và xử lý tọa độ PDF xoay 90/180/270°. Nếu batch cũ đã có các lỗi này, chọn
`RESULTS_DIR` mới (ví dụ `/kaggle/working/ocr_pdf_v80_fixed`), giữ `REFRESH_SOURCE=True`
và chạy từ đầu notebook để lấy code mới. Không resume ledger của code cũ.

Resume trong cùng phiên giữ nguồn/runtime và snapshot; thay input, keyword, model, code hoặc `ASSETS`
thì chọn thư mục kết quả mới. ZIP chứa kết quả/log/SQLite, **không chứa runtime, model hay checkpoint GPU**;
không bảo đảm resume sau khi phiên Kaggle bị xóa. Save Version/tải ZIP trước khi kết thúc phiên.

Notebook được kiểm tra bằng test cục bộ; chưa chạy trong một phiên Kaggle thực. `NEEDS_REVIEW`
cần đối chiếu PDF; kiểm artifact/Excel không phải chứng nhận độ chính xác OCR.
Global Alias Rules: literal, alias đã biết và Pending nằm riêng trong ma trận. Không coi Pending là 0 hoặc là kết quả đã nghiệm thu. `results.zip` chứa hàng đợi mới, rules snapshot và hash trong SQLite.


# 1. Cấu hình

Dùng `SOURCE_ARCHIVE` để thử đúng checkout chưa phát hành; đặt SHA-256 từ lệnh build. `PREVIOUS_RESULTS_DIR` nhận cả gói đã giải nén của phiên trước, không chỉ SQLite.


In [ ]:
from pathlib import Path
import collections
import hashlib
import io
import json
import os
import queue
import shutil
import signal
import subprocess
import sys
import threading
import time
import unicodedata
from zipfile import ZipFile, ZIP_DEFLATED

if sys.platform == "win32":
    for stream in (sys.stdout, sys.stderr):
        if hasattr(stream, "reconfigure"):
            stream.reconfigure(encoding="utf-8", errors="replace")

REPO_URL = "https://github.com/VuTuanAn1403/OCR_pdf.git"
REPO_REF = "main"                     # Nhánh/tag; checkout được khóa khi batch đã bắt đầu.
REFRESH_SOURCE = True                  # Chỉ cập nhật checkout trước khi batch có SQLite ledger.
SYNC_GPU_ENVS = False                  # True để sửa môi trường cài dang dở từ dependency lock.
INPUT_ROOT = Path("/kaggle/input")
PDF_FILES = []                         # []: tất cả; hoặc ["dataset/bao_cao.pdf"].
FILE_LIMIT = None                      # Không giới hạn; thử 1 file: đặt 1 và đổi RESULTS_DIR.
SOURCE_ARCHIVE = None                 # ZIP từ scripts/build_kaggle_source.py, gắn qua Add Input.
SOURCE_ARCHIVE_SHA256 = None           # SHA-256 do lệnh build in ra.
PREVIOUS_RESULTS_DIR = None            # Output phiên trước đã giải nén; không phải thư mục dataset PDF.
ALIAS_RULES_FILE = None                # None: rules cùng bản mã; hoặc đường dẫn JSON riêng.
GLOBAL_ALIASES = True                  # False: chỉ literal/occurrence, không dùng global rules.
KEYWORDS_FILE = None                   # None: bộ v1 của repository.
KEYWORD_SET_VERSION = "v1"             # Bộ mặc định v1 hiện có 86 display/86 canonical.
ASSETS = "none"                        # jpeg | png | none
RUN_FULL_BATCH = True
RESUME = True
RETRY_FAILED = False                   # True để thử lại tài liệu có trạng thái lỗi.
FORCE_OCR = False                      # True để OCR lại; giữ cùng input/runtime/output options.
RECOUNT_ONLY = False                   # Chỉ đếm lại batch v8.0 đã có, dùng snapshot keyword cũ.
MAX_ATTEMPTS = 2

RUNTIME_ROOT = Path("/tmp/ocr_pdf_v80_kaggle")
RESULTS_DIR = Path("/kaggle/working/ocr_pdf_v80_results")
LOG_DIR = RESULTS_DIR / "logs"
BATCH_DIR = RESULTS_DIR / "documents"
ARCHIVE_PATH = RESULTS_DIR.parent / "results.zip"

# Chạy lại cấu hình cần chuẩn bị lại input trước khi cài runtime.
INPUTS_READY = False


## 2. Log, kiểm input và đóng gói kết quả

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for part in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(part)
    return digest.hexdigest()


def write_frozen_json(path, payload, *, replace_existing=False):
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8")) == payload:
            return
        if not replace_existing:
            raise ValueError(f"Snapshot đã đổi: {path}. Chọn RESULTS_DIR mới.")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def run_logged(command, log_path, *, cwd=None, env=None, echo=True):
    command = [str(value) for value in command]
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Chạy: {command[0]} | Log: {log_path}", flush=True)
    lines = queue.Queue()
    tail = collections.deque(maxlen=25)
    last_progress = time.monotonic()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace",
            bufsize=1, start_new_session=(os.name == "posix"))

        def reader():
            try:
                for line in process.stdout:
                    lines.put(line)
            finally:
                lines.put(None)

        thread = threading.Thread(target=reader, daemon=True)
        thread.start()
        try:
            while True:
                try:
                    line = lines.get(timeout=1)
                except queue.Empty:
                    line = ""
                if line is None:
                    break
                if line:
                    log.write(line)
                    log.flush()
                    tail.append(line)
                    if echo:
                        print(line, end="", flush=True)
                        last_progress = time.monotonic()
                if time.monotonic() - last_progress >= 30:
                    print(f"Vẫn đang chạy; xem {log_path}", flush=True)
                    last_progress = time.monotonic()
            return_code = process.wait()
        except BaseException:
            if process.poll() is None:
                # SIGINT cho Python supervisor chạy cleanup của cây worker trước.
                if os.name == "posix":
                    os.killpg(process.pid, signal.SIGINT)
                else:
                    process.terminate()
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    if os.name == "posix":
                        os.killpg(process.pid, signal.SIGKILL)
                    else:
                        process.kill()
                    process.wait()
            raise
        finally:
            thread.join(timeout=2)
            process.stdout.close()
    if return_code:
        print("".join(tail))
        raise subprocess.CalledProcessError(return_code, command)
    if not echo:
        print("".join(list(tail)[-5:]))
    return log_path


def select_pdfs(root, requested=(), limit=None):
    root = Path(root).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Không có {root}. Hãy Add Input dataset PDF.")
    if limit is not None and (type(limit) is not int or limit < 1):
        raise ValueError("FILE_LIMIT phải là số nguyên dương hoặc None.")
    if requested:
        paths = [(Path(p) if Path(p).is_absolute() else root / p).resolve() for p in requested]
        if any(not p.is_file() or p.suffix.casefold() != ".pdf" or not p.is_relative_to(root) for p in paths):
            raise ValueError("PDF_FILES phải nằm trong INPUT_ROOT và có đuôi PDF.")
    else:
        paths = [p.resolve() for p in root.rglob("*") if p.is_file() and p.suffix.casefold() == ".pdf"]
        if any(not p.is_relative_to(root) for p in paths):
            raise ValueError("PDF symlink không được trỏ ra ngoài INPUT_ROOT.")
    paths = sorted(set(paths), key=lambda p: unicodedata.normalize("NFC", str(p)).casefold())
    if not paths:
        raise FileNotFoundError(f"Không tìm thấy PDF trong {root}. Hãy Add Input dataset PDF.")
    return paths if limit is None else paths[:limit]


def stage_pdfs(paths, runtime_root):
    identity = [{"source": str(p), "size": p.stat().st_size, "sha256": sha256_file(p)} for p in paths]
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
    directory = Path(runtime_root) / "selected-inputs" / key
    directory.mkdir(parents=True, exist_ok=True)
    stems = collections.Counter(unicodedata.normalize("NFC", p.stem).casefold() for p in paths)
    mapping, used = [], set()
    for source, item in zip(paths, identity):
        stem = unicodedata.normalize("NFC", source.stem)
        suffix = "-" + hashlib.sha256(str(source).encode()).hexdigest()[:12] if stems[stem.casefold()] > 1 else ""
        name = stem + suffix + ".pdf"
        if name.casefold() in used:
            raise ValueError(f"Tên PDF trùng sau chuẩn hóa: {name}")
        used.add(name.casefold())
        target = directory / name
        if target.exists() or target.is_symlink():
            if not target.is_file() or sha256_file(target) != item["sha256"]:
                raise ValueError(f"Staged PDF đã thay đổi: {target}")
        else:
            try:
                target.symlink_to(source)
            except OSError:
                shutil.copy2(source, target)
        mapping.append({**item, "staged": str(target), "document_id": target.stem})
    return directory, mapping


def collect_diagnostics():
    repo = globals().get("REPO_DIR")
    if repo is None:
        return
    # OCR_CACHE_DIR (RUNTIME_ROOT/cache) holds v7.0 job checkpoints since P12;
    # the repository cache holds those of earlier runs.
    for base in (RUNTIME_ROOT / "cache/v62-jobs", Path(repo) / ".cache/v62-jobs", RUNTIME_ROOT / "cache/docling-structure"):
        if base.is_dir():
            for source in base.rglob("*"):
                if source.is_file() and (source.suffix == ".log" or source.name in {"scheduler.json", "exited.json", "text.exited.json", "structure.exited.json", "export.exited.json"}):
                    target = RESULTS_DIR / "diagnostics" / base.name / source.relative_to(base)
                    target.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(source, target)


def bundle_results(results_dir, archive_path):
    results_dir, archive_path = Path(results_dir).resolve(), Path(archive_path).resolve()
    if not results_dir.is_dir():
        raise FileNotFoundError(f"Chưa có kết quả: {results_dir}")
    if archive_path.is_relative_to(results_dir):
        raise ValueError("Đặt ZIP ngoài RESULTS_DIR để tránh tự đóng gói chính nó.")
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = archive_path.with_suffix(".zip.tmp")
    with ZipFile(temporary, "w", compression=ZIP_DEFLATED, compresslevel=4) as archive:
        for path in sorted(results_dir.rglob("*")):
            if path.is_symlink():
                raise ValueError(f"Không đóng gói symlink: {path}")
            if path.is_file() and not path.name.endswith(".tmp"):
                archive.write(path, path.relative_to(results_dir).as_posix())
    os.replace(temporary, archive_path)
    return archive_path

def prepare_source_archive(source, runtime, expected_sha256=None):
    source, runtime = Path(source).resolve(), Path(runtime).resolve()
    checksum = sha256_file(source)
    if expected_sha256 is not None and checksum != expected_sha256:
        raise ValueError("Source archive SHA-256 mismatch")
    target = runtime / ("source-archive-" + checksum[:16])
    with ZipFile(source) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError("Duplicate source archive member")
        manifest = json.loads(archive.read("kaggle-source-manifest.json"))
        files = manifest.get("files")
        if manifest.get("schema_version") != "kaggle-source-v1" or not isinstance(files, dict):
            raise ValueError("Unsupported source archive manifest")
        if set(names) != set(files) | {"kaggle-source-manifest.json"}:
            raise ValueError("Source archive members differ from manifest")
        for name in files:
            parts = name.split("/")
            if not name or "\\" in name or ":" in name or any(p in ("", ".", "..") for p in parts):
                raise ValueError("Unsafe source archive path")
            info = archive.getinfo(name)
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Source archive symlinks are forbidden")
            if hashlib.sha256(archive.read(name)).hexdigest() != files[name]:
                raise ValueError("Source archive file checksum mismatch")
        for marker in ("pyproject.toml", "config/profiles.yaml", "config/alias_rules.json", "src/ocr_runtime/paths.py"):
            if marker not in files:
                raise ValueError("Source archive is missing required runtime files")
        if target.exists():
            if any(not (target/name).is_file() or sha256_file(target/name) != digest for name, digest in files.items()):
                raise ValueError("Existing source snapshot changed; use a new RUNTIME_ROOT")
        else:
            target.mkdir(parents=True)
            for name in files:
                destination = target / name
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(archive.read(name))
    return target, manifest, checksum


def restore_previous_results(source, target):
    source, target = Path(source).resolve(), Path(target).resolve()
    if not source.is_dir() or source == target or target.is_relative_to(source) or source.is_relative_to(target):
        raise ValueError("Previous results must be a separate extracted output directory")
    if target.exists():
        raise ValueError("RESULTS_DIR already exists; restoration never merges or overwrites it")
    for name in ("run_settings.json", "selected_inputs.json", "documents/batch_state.sqlite", "documents/batch_config.json"):
        if not (source/name).is_file():
            raise ValueError("Incomplete previous result package: " + name)
    if any(p.is_symlink() for p in source.rglob("*")):
        raise ValueError("Previous results must not contain symlinks")
    # Copy the complete artifact/SQLite snapshot; a lone database cannot prove extraction.
    shutil.copytree(source, target)
    for p in source.rglob("*"):
        if p.is_file() and sha256_file(p) != sha256_file(target/p.relative_to(source)):
            raise ValueError("Previous result copy checksum mismatch")
    return json.loads((target/"run_settings.json").read_text(encoding="utf-8"))


def freeze_input_file(source, destination):
    source, destination = Path(source), Path(destination)
    checksum = sha256_file(source)
    if destination.exists():
        if sha256_file(destination) != checksum:
            raise ValueError("Frozen input changed; choose a new RESULTS_DIR")
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(source.read_bytes())
    return destination


def validated_previous_commit(settings):
    commit = settings.get("commit")
    if not isinstance(commit, str) or len(commit) != 40 or any(c not in "0123456789abcdef" for c in commit):
        raise ValueError("Previous commit must be an exact Git SHA; use the original source archive when available")
    return commit


## 3. Chọn PDF trước khi cài model

PDF được gom vào thư mục phẳng để phù hợp CLI v8.0. File trùng tên ở hai dataset được
thêm hậu tố ổn định; file nguồn không bị sửa. Snapshot có hash nội dung, không chỉ dựa vào tên file.

In [ ]:
INPUTS_READY = False

if ASSETS not in {"jpeg", "png", "none"}:
    raise ValueError("ASSETS phải là jpeg/png/none.")
if type(MAX_ATTEMPTS) is not int or MAX_ATTEMPTS < 1:
    raise ValueError("MAX_ATTEMPTS phải là số nguyên dương.")
if KEYWORDS_FILE is not None and not Path(KEYWORDS_FILE).expanduser().is_file():
    raise FileNotFoundError(f"Không có file keyword: {KEYWORDS_FILE}")
if ALIAS_RULES_FILE is not None and not Path(ALIAS_RULES_FILE).expanduser().is_file():
    raise FileNotFoundError(f"Không có file alias rules: {ALIAS_RULES_FILE}")
if SOURCE_ARCHIVE is not None and not Path(SOURCE_ARCHIVE).expanduser().is_file():
    raise FileNotFoundError(f"Không có source archive: {SOURCE_ARCHIVE}")
RESTORED_SETTINGS = None
if PREVIOUS_RESULTS_DIR is not None:
    RESTORED_SETTINGS = restore_previous_results(PREVIOUS_RESULTS_DIR, RESULTS_DIR)
PDF_PATHS = select_pdfs(INPUT_ROOT, PDF_FILES, FILE_LIMIT)
STAGED_INPUT, INPUT_MAPPING = stage_pdfs(PDF_PATHS, RUNTIME_ROOT)
write_frozen_json(RESULTS_DIR / "selected_inputs.json", INPUT_MAPPING)
print(f"Đã chọn {len(INPUT_MAPPING)} PDF; v8.0 sẽ OCR toàn bộ trang của từng tài liệu.")
for item in INPUT_MAPPING[:20]:
    print(item["document_id"], "←", item["source"])
if len(INPUT_MAPPING) > 20:
    print(f"... và {len(INPUT_MAPPING) - 20} tài liệu khác")

INPUTS_READY = True


## 4. Checkout và cài ba môi trường riêng

Cell này chỉ chạy khi có GPU NVIDIA. Hai worker được cài từ **Linux lock có SHA-256**,
không dùng Torch/ONNX/Docling sẵn có trong kernel. `REFRESH_SOURCE=True` cập nhật checkout
**chỉ khi batch chưa có SQLite ledger**; giữ nguyên code khi batch đã bắt đầu.
Nếu phiên trước cài dependency dang dở, đặt `SYNC_GPU_ENVS=True` để đồng bộ lại từ lock.
Xem `logs/setup-gpu.log` nếu cài đặt thất bại.

Cell 3 phải chạy thành công trước cell này. Sau khi khởi động lại kernel hoặc đổi cấu hình ở cell 1, chọn **Run All** từ đầu.


In [ ]:
if not globals().get("INPUTS_READY", False):
    raise RuntimeError("Chưa chuẩn bị xong input. Chạy cell 1 (cấu hình), cell 2 (hàm hỗ trợ), "
                       "cell 3 (chọn PDF) đến khi in số PDF đã chọn, rồi chạy cell 4. "
                       "Sau Restart Session, hãy dùng Run All từ đầu notebook.")

if sys.platform != "linux":
    raise RuntimeError("Cell cài đặt này dành cho Kaggle/Linux. Dùng README cho Windows.")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Bật GPU trong Kaggle Settings rồi chạy lại.")
run_logged(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
           LOG_DIR / "gpu.log")

SOURCE_ARCHIVE_ACTUAL_SHA256 = None
if SOURCE_ARCHIVE is not None:
    REPO_DIR, SOURCE_MANIFEST, SOURCE_ARCHIVE_ACTUAL_SHA256 = prepare_source_archive(SOURCE_ARCHIVE, RUNTIME_ROOT, SOURCE_ARCHIVE_SHA256)
    SOURCE_COMMIT = SOURCE_MANIFEST["git_head"]
else:
    source_key = hashlib.sha256((REPO_URL + "\n" + REPO_REF).encode()).hexdigest()[:12]
    REPO_DIR = RUNTIME_ROOT / ("source-" + source_key)
    if not (REPO_DIR / ".git").is_dir():
        if REPO_DIR.exists():
            raise RuntimeError(f"Thư mục không phải Git checkout: {REPO_DIR}")
        run_logged(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO_DIR], LOG_DIR / "clone.log", echo=False)
    origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    if origin.rstrip("/") != REPO_URL.rstrip("/"):
        raise ValueError("Checkout có remote khác REPO_URL; chọn RUNTIME_ROOT mới.")
    if REFRESH_SOURCE and not (BATCH_DIR / "batch_state.sqlite").exists():
        run_logged(["git", "-C", REPO_DIR, "fetch", "origin", REPO_REF], LOG_DIR / "refresh-source.log", echo=False)
        run_logged(["git", "-C", REPO_DIR, "merge", "--ff-only", "FETCH_HEAD"], LOG_DIR / "update-source.log", echo=False)
    if globals().get("RESTORED_SETTINGS") is not None:
        pinned = validated_previous_commit(RESTORED_SETTINGS)
        run_logged(["git", "-C", REPO_DIR, "fetch", "origin", pinned], LOG_DIR / "restore-source.log", echo=False)
        run_logged(["git", "-C", REPO_DIR, "checkout", "--detach", "FETCH_HEAD"], LOG_DIR / "restore-checkout.log", echo=False)
    SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if not (REPO_DIR / "config/profiles.yaml").is_file():
    raise RuntimeError("REPO_REF chưa có pipeline v8.0.")
print("Commit:", SOURCE_COMMIT)

uv = shutil.which("uv")
if uv is None:
    bootstrap = RUNTIME_ROOT / "uv-bootstrap"
    run_logged([sys.executable, "-m", "pip", "install", "--ignore-installed", "--no-deps", "--prefix", bootstrap, "uv==0.12.10"],
               LOG_DIR / "install-uv.log", echo=False)
    uv = str(bootstrap / "bin/uv")
if not Path(uv).is_file():
    raise RuntimeError("Không tìm thấy uv; xem install-uv.log.")

RUNTIME_ENV = os.environ.copy()
for key in ("PYTHONPATH", "OCR_CONFIG_PATH", "OCR_DOCLING_PYTHON", "HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"):
    RUNTIME_ENV.pop(key, None)
MODELS_DIR = REPO_DIR / "models"
CACHE_DIR = RUNTIME_ROOT / "cache"
GPU_ENV_ROOT = REPO_DIR / ".venv-v62"
SUPERVISOR_PYTHON = REPO_DIR / ".venv/bin/python"
RUNTIME_ENV.update({
    "PATH": str(Path(uv).parent) + os.pathsep + RUNTIME_ENV.get("PATH", ""),
    "PYTHONNOUSERSITE": "1", "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1",
    "UV_CACHE_DIR": str(RUNTIME_ROOT / "uv-cache"), "UV_PYTHON_INSTALL_DIR": str(RUNTIME_ROOT / "python"),
    "HF_HOME": str(MODELS_DIR / "docling/huggingface"), "OCR_MODELS_DIR": str(MODELS_DIR),
    "VIETOCR_MODEL_DIR": str(MODELS_DIR / "vietocr"), "OCR_CACHE_DIR": str(CACHE_DIR),
    "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2", "TOKENIZERS_PARALLELISM": "false",
})
if not SUPERVISOR_PYTHON.is_file():
    run_logged([uv, "venv", "--python", "3.11", REPO_DIR / ".venv"], LOG_DIR / "python-311.log", env=RUNTIME_ENV, echo=False)
run_logged([uv, "pip", "install", "--python", SUPERVISOR_PYTHON,
    "pymupdf>=1.24,<=1.28.2", "pydantic>=2,<=2.14.0", "PyYAML>=6,<=6.0.3",
    "psutil>=5.9,<=7.2.2", "openpyxl>=3.1,<4", "tabulate>=0.9,<=0.10.0"],
    LOG_DIR / "setup-supervisor.log", env=RUNTIME_ENV, echo=False)
setup_gpu_command = [SUPERVISOR_PYTHON, "scripts/setup_v62.py", "--python", "3.11", "--env-root", GPU_ENV_ROOT]
if SYNC_GPU_ENVS:
    setup_gpu_command.append("--sync")
run_logged(setup_gpu_command,
           LOG_DIR / "setup-gpu.log", cwd=REPO_DIR, env=RUNTIME_ENV, echo=False)
KEYWORDS_PATH = Path(KEYWORDS_FILE).expanduser().resolve() if KEYWORDS_FILE else REPO_DIR / "config/keywords.txt"
RULES_PATH = (Path(ALIAS_RULES_FILE).expanduser().resolve() if ALIAS_RULES_FILE else REPO_DIR / "config/alias_rules.json") if GLOBAL_ALIASES else None
if RULES_PATH is not None and not RULES_PATH.is_file():
    raise FileNotFoundError("Bản mã chưa có alias rules; dùng SOURCE_ARCHIVE mới hoặc tắt GLOBAL_ALIASES.")
if RULES_PATH is not None:
    RULES_PATH = freeze_input_file(RULES_PATH, RESULTS_DIR / "alias_rules.json")
freeze_input_file(KEYWORDS_PATH, RESULTS_DIR / "keywords.txt")
write_frozen_json(RESULTS_DIR / "run_settings.json", {
    "repository": REPO_URL, "ref": REPO_REF, "commit": SOURCE_COMMIT, "backend": "gpu_two_phase",
    "profile": "unified_gpu_full_ocr", "assets": ASSETS, "output_mode": "compact",
    "keywords_file": str(KEYWORDS_PATH), "keyword_set_version": KEYWORD_SET_VERSION,
    "keywords_sha256": sha256_file(KEYWORDS_PATH), "selected_files": len(INPUT_MAPPING),
    "alias_rules_file": None if RULES_PATH is None else str(RULES_PATH),
    "alias_rules_sha256": None if RULES_PATH is None else sha256_file(RULES_PATH),
    "source_archive_sha256": SOURCE_ARCHIVE_ACTUAL_SHA256,
}, replace_existing=not (BATCH_DIR / "batch_state.sqlite").exists())

## 5. Tải model đúng hash/revision

VietOCR dùng script tải của dự án (`vgg_transformer` v8.0 và `vgg_seq2seq`). Detector (`PP-OCRv6_medium_det`) và
Layout (`PP-DocLayout_plus-L`) ONNX tải đúng repo/revision và SHA-256 trong `models/v62/runtime-models.json`.
Snapshot Docling được tải đúng revision trong `models/v62/runtime-models.json`; kiểm toàn bộ hash trước khi chạy. Không viết lại registry để che lỗi hash.

In [ ]:
DETECTOR_DOWNLOAD_PROGRAM = r"""
import hashlib, io, json
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
registry = json.loads(Path('models/v62/runtime-models.json').read_text(encoding='utf-8'))
for role in ('text_detector', 'layout_model'):
    entry = registry.get(role)
    if not isinstance(entry, dict):
        continue
    target = Path(entry['model_path'])
    expected = entry['sha256']
    if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
        source = entry.get('source')
        if isinstance(source, dict) and source.get('repo') and source.get('revision') and source.get('file'):
            url = f"https://huggingface.co/{source['repo']}/resolve/{source['revision']}/{source['file']}"
            with urlopen(url, timeout=180) as response:
                data = response.read()
        elif role == 'text_detector':
            with urlopen('https://pypi.org/pypi/rapidocr-onnxruntime/1.4.4/json', timeout=60) as response:
                metadata = json.load(response)
            wheels = sorted((item for item in metadata['urls'] if item['packagetype'] == 'bdist_wheel'), key=lambda item: item['filename'])
            if not wheels: raise RuntimeError('Không tìm thấy wheel detector')
            wheel = wheels[0]
            with urlopen(wheel['url'], timeout=120) as response:
                payload = response.read()
            if hashlib.sha256(payload).hexdigest() != wheel['digests']['sha256']:
                raise ValueError('Wheel download checksum mismatch')
            with ZipFile(io.BytesIO(payload)) as archive:
                candidates = [name for name in archive.namelist() if name.endswith('/models/ch_PP-OCRv4_det_infer.onnx')]
                if len(candidates) != 1: raise ValueError('Detector không duy nhất trong wheel')
                data = archive.read(candidates[0])
        else:
            raise ValueError(f'Missing pinned source for {role}')
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError(f'{role} khác hash registry')
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = target.with_suffix('.onnx.download')
        temporary.write_bytes(data)
        temporary.replace(target)
    print('ONNX model checksum OK:', role, target)
"""

DOCLING_DOWNLOAD_PROGRAM = r"""
import hashlib, json
from pathlib import Path
from huggingface_hub import snapshot_download
registry = json.loads(Path('models/v62/runtime-models.json').read_text(encoding='utf-8'))
hub = Path('models/docling/huggingface/hub').resolve()
for role, model in registry['snapshots'].items():
    prefix = 'models/docling/huggingface/hub/models--' + model['repo'].replace('/', '--') + '/snapshots/' + model['revision'] + '/'
    files = {name: checksum for name, checksum in registry['files'].items() if name.startswith(prefix)}
    valid = files and all(Path(name).is_file() and hashlib.sha256(Path(name).read_bytes()).hexdigest() == checksum for name, checksum in files.items())
    if not valid:
        snapshot_download(model['repo'], revision=model['revision'], cache_dir=str(hub))
    ref = hub / ('models--' + model['repo'].replace('/', '--')) / 'refs' / ('v2.3.0' if role == 'table' else 'main')
    ref.parent.mkdir(parents=True, exist_ok=True)
    ref.write_text(model['revision'], encoding='utf-8')
    print('Snapshot:', role, model['revision'], flush=True)
for name, expected in registry['files'].items():
    path = Path(name)
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError('MODEL_LOCK_MISMATCH: ' + name)
print('Toàn bộ model đúng SHA-256 registry.')
"""

run_logged([SUPERVISOR_PYTHON, "scripts/setup_vietocr_model.py"], LOG_DIR / "download-vietocr.log", cwd=REPO_DIR, env=RUNTIME_ENV, echo=False)
run_logged([SUPERVISOR_PYTHON, "-c", DETECTOR_DOWNLOAD_PROGRAM], LOG_DIR / "download-detector.log", cwd=REPO_DIR, env=RUNTIME_ENV)
run_logged([GPU_ENV_ROOT / "docling/bin/python", "-c", DOCLING_DOWNLOAD_PROGRAM], LOG_DIR / "download-docling.log", cwd=REPO_DIR, env=RUNTIME_ENV, echo=False)

## 6. Kiểm CUDA, provider và keyword trước khi OCR

In [ ]:
run_logged([SUPERVISOR_PYTHON, "scripts/preflight_v62.py", "--device", "cuda", "--env-root", GPU_ENV_ROOT,
    "--output", RESULTS_DIR / "gpu-preflight.json"], LOG_DIR / "preflight.log", cwd=REPO_DIR, env=RUNTIME_ENV)
probe = r"""
import json, sys
from src.ocr.application.process_document import finalize_keywords
from src.ocr.infrastructure.postprocessing.keyword_contract import dictionary_snapshot
snapshot = dictionary_snapshot(sys.argv[1], sys.argv[2])
if 'torch' in sys.modules: raise RuntimeError('Supervisor đã nạp Torch')
print(json.dumps({'python':sys.version,'supervisor_imported_torch':False,
 'display_keywords':len(snapshot['display_columns']),'canonical_keywords':len(snapshot['canonical_keywords'])}))
"""
run_logged([SUPERVISOR_PYTHON, "-c", probe, KEYWORDS_PATH, KEYWORD_SET_VERSION], LOG_DIR / "supervisor-probe.log", cwd=REPO_DIR, env=RUNTIME_ENV)
font_probe = "from src.ocr.infrastructure.pdf.ocr_bridge import load_bridge_font; p, f = load_bridge_font(); print('OCR bridge font:', p, f.name)"
run_logged([GPU_ENV_ROOT / "text/bin/python", "-c", font_probe], LOG_DIR / "bridge-font.log", cwd=REPO_DIR, env=RUNTIME_ENV)

## 7. OCR toàn bộ PDF → Excel riêng → ma trận chung

Batch cập nhật `documents/MA_TRAN_DEM_TU_KHOA_<N>_FILES.xlsx` sau từng PDF.
Ô trống là đang chờ/lỗi; `0` nghĩa là đã đếm và không gặp keyword. Bộ v1 có 88 cột/87 cụm duy nhất.
Không truyền `--pages`, `--limit`, `--hardware`, `--threads-per-worker` của CLI legacy.

Nếu cell báo lỗi, ZIP kết quả/log vẫn được tạo trong `finally` khi có thể. Đóng workbook đang được ứng dụng khác
giữ, xem log rồi chạy lại với `RESUME = True`; muốn thử lại PDF đã lỗi, bật `RETRY_FAILED`.
Đổi từ điển/rules cần output analysis mới bằng `scripts/run_count_only.py --alias-rules ... --output ...`. RECOUNT_ONLY giữ snapshot cũ.


In [ ]:
def build_batch_command():
    if RECOUNT_ONLY:
        if not (BATCH_DIR / "batch_state.sqlite").is_file():
            raise FileNotFoundError("Chưa có batch v8.0 để recount.")
        return [str(SUPERVISOR_PYTHON), str(REPO_DIR / "run_batch.py"), "--output", str(BATCH_DIR), "--recount"]
    command = [str(SUPERVISOR_PYTHON), str(REPO_DIR / "run_batch.py"), "--backend", "gpu_two_phase",
        "--input", str(STAGED_INPUT), "--output", str(BATCH_DIR), "--env-root", str(GPU_ENV_ROOT),
        "--config", str(REPO_DIR / "config/profiles.yaml"), "--profile", "unified_gpu_full_ocr",
        "--device", "cuda", "--workers", "1", "--assets", ASSETS, "--output-mode", "compact",
        "--keywords-file", str(KEYWORDS_PATH), "--keyword-set-version", KEYWORD_SET_VERSION,
        "--max-attempts", str(MAX_ATTEMPTS)]
    if GLOBAL_ALIASES:
        command.extend(["--alias-rules", str(globals().get("RULES_PATH") or REPO_DIR / "config/alias_rules.json")])
    else:
        command.append("--no-alias-rules")
    if RESUME:
        command.append("--resume")
    if RETRY_FAILED:
        command.append("--retry-failed")
    if FORCE_OCR:
        command.append("--force-ocr")
    return command


if RUN_FULL_BATCH:
    try:
        run_logged(build_batch_command(), LOG_DIR / "batch-console.log", cwd=REPO_DIR, env=RUNTIME_ENV)
    finally:
        try:
            collect_diagnostics()
        finally:
            if RESULTS_DIR.is_dir():
                print("ZIP tạm/kết quả:", bundle_results(RESULTS_DIR, ARCHIVE_PATH))
else:
    print("Đã chuẩn bị runtime. Đặt RUN_FULL_BATCH=True rồi chạy cell này để bắt đầu OCR.")

## 8. Kiểm SQLite, receipt và Excel

Cell xác minh số tài liệu, revision SQLite, hash workbook và các artifact đã công bố. Số đếm được đọc
từ chính pipeline; không có bộ đếm thứ hai trong notebook. `NEEDS_REVIEW` vẫn hiển thị trong summary.

In [ ]:
VERIFY_PROGRAM = r"""
import hashlib, json, sqlite3, sys
from pathlib import Path
from src.ocr.infrastructure.export.completion import verify_completed
from src.ocr.infrastructure.export.artifact_validator import validate_document_artifacts
from src.ocr.infrastructure.export.keyword_v63 import verify_matrix
from src.ocr.infrastructure.postprocessing.keyword_contract import validate_report

root, summary_path = Path(sys.argv[1]), Path(sys.argv[3])
expected_ids = set(json.loads(sys.argv[2]))
snapshot = json.loads((root/'keyword_dictionary.snapshot.json').read_text(encoding='utf-8'))
report = json.loads((root/'batch_report.json').read_text(encoding='utf-8'))
connection = sqlite3.connect((root/'batch_state.sqlite').resolve().as_uri() + '?mode=ro', uri=True)
try:
    meta = {key:json.loads(value) for key,value in connection.execute('SELECT key,value FROM meta')}
    records = [json.loads(row[0]) for row in connection.execute('SELECT payload FROM documents ORDER BY ordinal')]
    revision = meta['published_revision']
    row = connection.execute('SELECT projection,workbook_sha256 FROM revisions WHERE revision=?', (revision,)).fetchone()
finally:
    connection.close()
if meta['prepared_revision'] != revision or report['published_revision'] != revision or report['documents'] != records:
    raise ValueError('Revision chưa công bố hoặc report khác SQLite; chạy lại resume.')
if {record['id'] for record in records} != expected_ids or len(records) != len(expected_ids):
    raise ValueError('Danh sách output không khớp input đã chọn')
if json.loads(row[0]) != records or report['batch_id'] != meta['identity']['batch_id']:
    raise ValueError('Projection/batch identity không khớp')
matrix = root / f'MA_TRAN_DEM_TU_KHOA_{len(records)}_FILES.xlsx'
if hashlib.sha256(matrix.read_bytes()).hexdigest() != row[1]:
    raise ValueError('Ma trận khác hash SQLite')
rendered = records
queue = None
if meta['identity'].get('config', {}).get('alias_rules') is not None:
    from ocr_pipeline.alias_finalization import publication_material
    rendered, queue = publication_material(records, snapshot, meta['identity']['config'], report['batch_id'], revision)
    queue_path = root/'new_suspicions_queue.json'
    if json.loads(queue_path.read_text(encoding='utf-8')) != queue:
        raise ValueError('Pending queue differs from the published revision')
    if hashlib.sha256(queue_path.read_bytes()).hexdigest() != meta.get('new_suspicions_queue_sha256'):
        raise ValueError('Pending queue SHA-256 differs from SQLite')
verify_matrix(matrix, report['batch_id'], revision, rendered, snapshot)
documents = []
for record in records:
    if record.get('outcome') != 'SUCCEEDED':
        raise ValueError('PDF chưa thành công: ' + record['id'] + ': ' + str(record.get('error')))
    directory = root / record['id']
    if Path(record['output']).resolve() != directory.resolve() or not verify_completed(directory):
        raise ValueError('Output thiếu/hỏng: ' + str(directory))
    if hashlib.sha256((directory/'COMPLETED.json').read_bytes()).hexdigest() != record['completion_sha256']:
        raise ValueError('Completion receipt khác SQLite')
    counts_path = directory/'keyword_counts.json'
    if hashlib.sha256(counts_path.read_bytes()).hexdigest() != record['report_sha256']:
        raise ValueError('Keyword report khác SQLite')
    counts = json.loads(counts_path.read_text(encoding='utf-8'))
    validate_report(counts, snapshot, directory/'result.md', record['id'], record['pdf_sha256'])
    validation = validate_document_artifacts(directory)
    if validation['status'] == 'FAIL': raise ValueError('Artifact validation thất bại: ' + record['id'])
    documents.append({'document_id':record['id'],'validation':validation['status'],
        'pages':record['total_pages'],'canonical_matches':counts['total_canonical_matches'],
        'display_matches':counts['total_display_matches']})
summary = {'batch_id':report['batch_id'],'revision':revision,'matrix':str(matrix),
    'documents':documents,'display_keywords':len(snapshot['display_columns']),
    'canonical_keywords':len(snapshot['canonical_keywords']),'quality_acceptance':'NOT_EVALUATED',
    'global_aliases':queue is not None,'pending_total':None if queue is None else queue['pending_total']}
summary_path.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))
"""

if RUN_FULL_BATCH:
    run_logged([SUPERVISOR_PYTHON, "-c", VERIFY_PROGRAM, BATCH_DIR,
        json.dumps([item["document_id"] for item in INPUT_MAPPING]), RESULTS_DIR / "verification.json"],
        LOG_DIR / "verification.log", cwd=REPO_DIR, env=RUNTIME_ENV, echo=False)
    SUMMARY = json.loads((RESULTS_DIR / "verification.json").read_text(encoding="utf-8"))
    MATRIX_PATH = Path(SUMMARY["matrix"])
    print("Đã kiểm tra:", len(SUMMARY["documents"]), "PDF; ma trận:", MATRIX_PATH)
else:
    print("Chưa chạy batch; chưa có ma trận để xác minh.")

## 9. Tải Excel và ZIP

Có thể chạy lại cell này sau lỗi để lấy log/kết quả đã có; ZIP có thể là batch chưa hoàn tất.
Trong Kaggle cũng có thể tải từ mục Output. Khi batch dài, lưu kết quả trước khi hết phiên.

In [ ]:
collect_diagnostics()
archive = bundle_results(RESULTS_DIR, ARCHIVE_PATH)
print(f"ZIP: {archive} ({archive.stat().st_size / 1024**2:.1f} MiB)")
from IPython.display import FileLink, display
def download_link(path):
    try:
        link = os.path.relpath(path, Path.cwd())
    except ValueError:  # Local Windows verification may use a different drive.
        link = Path(path).as_posix()
    return FileLink(link)

for matrix in sorted(BATCH_DIR.glob("MA_TRAN_DEM_TU_KHOA_*_FILES.xlsx")):
    print("Ma trận:", matrix)
    display(download_link(matrix))
display(download_link(archive))